In [ ]:
from IPython.display import Image as IPImage
from IPython.display import Image, display

from typing import TypedDict, Optional, List, Dict, Any, Annotated, Tuple, Optional, Literal, Callable
from typing_extensions import TypedDict
import operator

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, BaseMessage
from langchain_openai import ChatOpenAI

from langgraph.graph.message import add_messages

### NEW
from datetime import datetime, timedelta, date
import pandas as pd
from langchain.tools import tool
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from datetime import date, datetime, timedelta
import pandas as pd
from __future__ import annotations
from langchain_core.tools import tool
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import csv
import re
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import json
from langchain_core.documents import Document

load_dotenv(find_dotenv())

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRACING_V2") == "true"

KB_DIR = Path.cwd() / "kb"
BusinessMarketing_KB_PATH = KB_DIR / "BusinessMarketing"
MembershipFraud_KB_PATH = KB_DIR / "MembershipFraud"

import sqlite3
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.tools import tool

DB_PATH = MembershipFraud_KB_PATH / "membership_fraud.db"

CARDIOLOGY_SCHEDULE_CSV = KB_DIR / "cardiology_appointment_slots.csv"

CARDIOLOGY_KB_PATH = "./kb/cardiology_kb.jsonl"

REBUILD_CHROMA = False   # <-- set to False to reuse persisted DB
CHROMA_DIR = "./chroma_kb"
#CHROMA_DIR = "./chroma_kb_rebuild" if REBUILD_CHROMA else "./chroma_kb"

print("LANGCHAIN_PROJECT:", LANGCHAIN_PROJECT)
print("LANGCHAIN_TRACING_V2:", LANGCHAIN_TRACING_V2)

In [ ]:
from __future__ import annotations

from typing import List, Optional, TypedDict, Annotated, Literal, Dict, Any
from datetime import datetime, timedelta

from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

from langchain_openai import ChatOpenAI


# ----------------------------
# Helper functions
# ----------------------------

def user_says_not_recognized(text: str) -> bool:
    t = text.lower()
    return any(
        phrase in t
        for phrase in [
            "don't recognize",
            "do not recognize",
            "wasn't me",
            "was not me",
            "not me",
            "i didn't do that",
        ]
    )

def user_says_recognized(text: str) -> bool:
    t = text.lower()
    return any(
        phrase in t
        for phrase in [
            "that was me",
            "i recognize",
            "it was me",
            "yes that was me",
        ]
    )


# ----------------------------
# Tool arg schemas
# ----------------------------

class ReadSecurityEventsInput(BaseModel):
    member_id: str = Field(..., description="Member identifier like M001")
    timeframe: Literal["most_recent", "last_7_days", "last_30_days"] = Field(
        default="most_recent",
        description="How far back to look"
    )
    max_events: int = Field(default=5, ge=1, le=25, description="Max events to return")

class ExplainSecurityEventInput(BaseModel):
    event: Dict[str, Any]

class GuideSecurityActionsInput(BaseModel):
    event: Dict[str, Any]


@tool(args_schema=ReadSecurityEventsInput)
def read_security_events(member_id: str, timeframe: str = "most_recent", max_events: int = 5) -> List[Dict[str, Any]]:
    """Retrieve security login events from SQLite for a member within a timeframe."""
    # Pin to your assignment's 'today'. Swap to datetime.now().isoformat(timespec="seconds") later if desired.
    if timeframe == "most_recent":
        sql = """
        SELECT event_id, member_id, event_ts, login_location, device_type, risk_level, trigger_reason, recommended_action
        FROM security_events
        WHERE member_id = ?
        ORDER BY event_ts DESC
        LIMIT ?
        """
        params = (member_id, max_events)
    else:
        cutoff = "2026-02-24T12:00:00" if timeframe == "last_7_days" else "2026-02-01T12:00:00"
        sql = """
        SELECT event_id, member_id, event_ts, login_location, device_type, risk_level, trigger_reason, recommended_action
        FROM security_events
        WHERE member_id = ? AND event_ts >= ?
        ORDER BY event_ts DESC
        LIMIT ?
        """
        params = (member_id, cutoff, max_events)

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(sql, params)
    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows

@tool(args_schema=ExplainSecurityEventInput)
def explain_security_event(event: Dict[str, Any]) -> str:
    """
    Generate a plain-language explanation of why a login event was flagged.

    Uses only the fields provided in the event record:
    - event_ts
    - login_location
    - device_type
    - trigger_reason
    - risk_level

    Does NOT expose internal fraud detection rules beyond trigger_reason.
    """
    return (
        f"Alert time: {event.get('event_ts')}\n"
        f"Location: {event.get('login_location')}\n"
        f"Device: {event.get('device_type')}\n"
        f"Flag reason: {event.get('trigger_reason')}\n"
        f"Risk level: {event.get('risk_level')}"
    )


@tool(args_schema=GuideSecurityActionsInput)
def guide_security_actions(event: Dict[str, Any]) -> List[str]:
    """
    Return a deterministic checklist of recommended security actions
    based on the event's risk_level.

    - High: urgent steps including MFA + contact support
    - Medium: precautionary steps
    - Low: reassurance with optional review

    Does not escalate beyond the recommended_action policy.
    """
    risk = (event.get("risk_level") or "").lower()
    base = [
        "Review recent account activity and confirm whether you recognize this sign-in.",
        "If you don’t recognize it: log out of all sessions (if available) and change your password.",
    ]

    if risk == "high":
        return base + [
            "Enable multi-factor authentication (MFA).",
            "Contact support to secure the account and review next steps.",
        ]

    if risk == "medium":
        return base + [
            "Consider enabling MFA for added protection.",
            "Check that account recovery info is up to date.",
        ]

    return base + [
        "No urgent action is required if you recognize the device/location."
    ]


# ----------------------------
# Agentic planner schema (LLM outputs this)
# ----------------------------
class SecurityPlan(BaseModel):
    member_id: Optional[str] = None
    timeframe: Literal["most_recent", "last_7_days", "last_30_days"] = "most_recent"
    max_events: int = Field(default=1, ge=1, le=10)
    ask_for_member_id: bool = False
    tone: Literal["calm", "neutral", "urgent"] = "neutral"


# ----------------------------
# LangGraph state
# ----------------------------
class SecurityState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    member_id: Optional[str]
    timeframe: Optional[Literal["most_recent", "last_7_days", "last_30_days"]]
    plan: Optional[Dict[str, Any]]              # <-- store dict, not Pydantic model
    retrieved_events: Optional[List[Dict[str, Any]]]
    final_response: Optional[str]


def _latest_user_text(state: SecurityState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            return m.content
    return ""


# ----------------------------
# Nodes
# ----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def plan_node(state: SecurityState) -> SecurityState:
    user_text = _latest_user_text(state)

    # carry forward from state if already known
    existing_member_id = state.get("member_id")
    existing_timeframe = state.get("timeframe")

    planner = llm.with_structured_output(SecurityPlan)

    plan_obj = planner.invoke(
        [
            HumanMessage(
                content=(
                    "You are a security support assistant.\n"
                    "Extract member_id and timeframe if present.\n"
                    "If member_id is missing BUT it is provided as EXISTING_MEMBER_ID, reuse it.\n"
                    "If after that member_id is still missing, set ask_for_member_id=true.\n"
                    "Timeframe mapping:\n"
                    "- 'latest/most recent' -> most_recent\n"
                    "- 'last week/past 7 days' -> last_7_days\n"
                    "- 'last month/past 30 days' -> last_30_days\n\n"
                    f"EXISTING_MEMBER_ID: {existing_member_id}\n"
                    f"EXISTING_TIMEFRAME: {existing_timeframe}\n\n"
                    f"USER: {user_text}"
                )
            )
        ]
    )

    # IMPORTANT: store as dict to avoid Pydantic serialization warnings
    plan = plan_obj.model_dump()

    # Persist member_id/timeframe in state for next turns
    member_id = plan.get("member_id") or existing_member_id
    timeframe = plan.get("timeframe") or existing_timeframe

    # If planner didn’t include member_id but we have it, don’t ask again
    if member_id:
        plan["ask_for_member_id"] = False
        plan["member_id"] = member_id

    return {
        "plan": plan,
        "member_id": member_id,
        "timeframe": timeframe,
    }

def maybe_ask_member_id(state: SecurityState) -> SecurityState:
    plan = state.get("plan") or {}

    # plan is a dict now
    if plan.get("ask_for_member_id", False) and not state.get("member_id"):
        msg = "I can look that up — what is your member_id (e.g., `M001`)?"
        return {"messages": [AIMessage(content=msg)], "final_response": msg}

    return {}

def retrieve_node(state: SecurityState) -> SecurityState:
    member_id = state["member_id"]
    timeframe = state.get("timeframe") or "most_recent"
    plan = state.get("plan") or {}
    max_events = plan.get("max_events", 1)

    events = read_security_events.invoke(
        {"member_id": member_id, "timeframe": timeframe, "max_events": max_events}
    )
    return {"retrieved_events": events}

def response_node(state: SecurityState) -> SecurityState:
    plan = state.get("plan") or {}
    events = state.get("retrieved_events") or []
    member_id = state.get("member_id")

    if not member_id:
        return {}

    if not events:
        msg = f"I couldn’t find any security alerts for {member_id} in that timeframe."
        return {"messages": [AIMessage(content=msg)], "final_response": msg}

    latest_event = events[0]
    explanation = explain_security_event.invoke({"event": latest_event})
    actions = guide_security_actions.invoke({"event": latest_event})

    user_text = _latest_user_text(state)

    recognized = user_says_recognized(user_text)
    not_recognized = user_says_not_recognized(user_text)

    writer = llm  # reuse your ChatOpenAI instance

    if not_recognized:
        # Escalation response (no repeated question)
        msg = writer.invoke([
            HumanMessage(
                content=(
                    "You are a security support assistant in a live chat.\n"
                    "Respond conversationally.\n"
                    "No subject line. No greetings. No signatures. No letter format.\n"
                    "The user says they do NOT recognize the login.\n"
                    "Be clear and action-oriented.\n"
                    "Use only the facts below.\n\n"
                    f"EVENT DETAILS:\n{explanation}\n\n"
                    f"RECOMMENDED ACTIONS:\n- " + "\n- ".join(actions)
                )
            )
        ]).content

    elif recognized:
        # Reassurance response
        msg = writer.invoke([
            HumanMessage(
                content=(
                    "You are a security support assistant in a live chat.\n"
                    "Respond conversationally.\n"
                    "No subject line. No greetings. No signatures.\n"
                    "The user confirms they recognize the login.\n"
                    "Reassure appropriately based on risk level.\n"
                    "Use only the facts below.\n\n"
                    f"EVENT DETAILS:\n{explanation}\n"
                )
            )
        ]).content

    else:
        # First-time explanation — ask follow-up once
        msg = writer.invoke([
            HumanMessage(
                content=(
                    "You are a security support assistant in a live chat.\n"
                    "Respond conversationally.\n"
                    "No subject line. No greetings. No signatures.\n"
                    "Provide the alert details and recommended actions.\n"
                    "End with one short question asking if they recognize the location or device.\n"
                    "Use only the facts below.\n\n"
                    f"EVENT DETAILS:\n{explanation}\n\n"
                    f"RECOMMENDED ACTIONS:\n- " + "\n- ".join(actions)
                )
            )
        ]).content

    return {"messages": [AIMessage(content=msg)], "final_response": msg}

def route_after_plan(state: SecurityState) -> str:
    # If member_id is known, proceed. Otherwise ask.
    return "ask" if not state.get("member_id") else "go"


# ----------------------------
# Build graph
# ----------------------------
builder = StateGraph(SecurityState)

builder.add_node("plan", plan_node)
builder.add_node("ask_member_id", maybe_ask_member_id)
builder.add_node("retrieve", retrieve_node)
builder.add_node("respond", response_node)

builder.set_entry_point("plan")

builder.add_conditional_edges(
    "plan",
    route_after_plan,
    {"ask": "ask_member_id", "go": "retrieve"},
)

builder.add_edge("ask_member_id", END)
builder.add_edge("retrieve", "respond")
builder.add_edge("respond", END)

security_graph = builder.compile()

In [ ]:
display(Image(security_graph.get_graph().draw_mermaid_png()))

<br><br><br>

<hr style="border:30px solid coral "> </hr>
<hr style="border:2px solid coral "> </hr>


# Demonstration

<hr style="border:2px solid coral "> </hr>


In [ ]:
result = security_graph.invoke(
    {
        "messages": [
            HumanMessage(content="Member M001, show me suspicious logins from last 14 days.")
        ]
    }
)

print(result["messages"][-1].content)

In [ ]:
print("Security Assistant ready. Type 'exit' to quit.\n")

state = {"messages": [], "member_id": None, "timeframe": None, "plan": None, "retrieved_events": None, "final_response": None}

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    state = security_graph.invoke({
        **state,
        "messages": state["messages"] + [
            HumanMessage(content=user_input)
        ]
    })

    print("\nAssistant:")
    print(state["messages"][-1].content)
    print()